# MeatGuard II — 00 · Teacher training and selection

Trains the knowledge-distillation teacher for MeatGuard-S and selects it without using the test set.

- Inputs: SAM-cropped MeatScan images (`crops/`) split by capture session (`sessions.csv`).
- Candidates: ImageNet-pretrained `efficientnet_b0`, `convnext_tiny`, `swin_t`, `resnet50`, same recipe for all.
- Selection: highest validation macro-F1 (ties: spoiled recall, then fewer parameters). The test split is evaluated
  once, for the selected teacher only, with 95% CIs that resample sessions.

All logic is in `src/train_teacher.py`; this notebook only drives it.

**Saved to `runs/teacher/`:** `<arch>/best.pt`, `last.pt`, `history.csv`, `summary.json` (settings, parameters,
best validation metrics, GPU and versions) and `teacher_selection.json` (validation ranking + test metrics).

## 1. Setup
Upload the whole `MeatGuard_II` folder to Google Drive and the data zip (`crops/`, `masks/`, `sessions.csv`) next to
it, then edit the two paths below if yours differ. Results are written to `MeatGuard_II/runs/` on Drive, so they
survive disconnects; re-running a training cell resumes from the last saved epoch.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- paths: edit these -------------------------------------------------------------------------
REPO_DIR = "/content/drive/MyDrive/MeatGuard_II" if IN_COLAB else os.path.abspath("..")   # the codebase on Drive
DATA_SRC = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "data/processed/full"  # zip or folder: crops/, masks/, sessions.csv
# -------------------------------------------------------------------------------------------------
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC     # local copy on the Colab VM (Drive is slow for many small files)
RUNS = os.path.join(REPO_DIR, "runs")
WORKERS = 4 if IN_COLAB else 0
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if IN_COLAB:
    !pip -q install torch-pruning shap
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data

In [ ]:
# Copy the data to the VM once, then check it: every session in one partition, every listed image present
import zipfile, shutil
import pandas as pd
from pathlib import Path
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(DATA_ROOT)
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
if not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):      # zip made from the parent folder
    nested = [p.parent for p in Path(DATA_ROOT).glob("*/sessions.csv")]
    DATA_ROOT = str(nested[0]) if nested else DATA_ROOT
CROPS, SPLIT, MASKS = (os.path.join(DATA_ROOT, d) for d in ("crops", "sessions.csv", "masks"))

from src.train_teacher import check_split
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", check_split(SPLIT), "| masks/ present:", os.path.isdir(MASKS))
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"

In [ ]:
# A few training crops per class
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for r, cls in enumerate(sorted(table["class"].unique())):
    sample = table[(table["split"] == "train") & (table["class"] == cls)].sample(8, random_state=0)
    for ax, rel in zip(axes[r], sample["path"]):
        ax.imshow(Image.open(Path(CROPS) / rel)); ax.axis("off")
    axes[r, 0].set_title(cls, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 3. Smoke test
Three batches, one epoch: catches path, memory and download problems in a minute.

In [ ]:
!python -m src.train_teacher train --arch efficientnet_b0 --data {CROPS} --split-file {SPLIT} \
    --out /content/smoke --epochs 1 --batch-size 16 --num-workers {WORKERS} --smoke --restart

## 4. Train the candidates
Re-run this cell after a disconnect to resume.

In [ ]:
ARCHS = ["efficientnet_b0", "convnext_tiny", "swin_t", "resnet50"]
TEACHER_RUNS = os.path.join(RUNS, "teacher")
for arch in ARCHS:
    print("=" * 30, arch, "=" * 30)
    !python -m src.train_teacher train --arch {arch} --data {CROPS} --split-file {SPLIT} --out {TEACHER_RUNS} \
        --num-workers {WORKERS}

In [ ]:
import matplotlib.pyplot as plt
def plot_curves(run_dirs, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for d in run_dirs:
        h = Path(d) / "history.csv"
        if not h.exists():
            continue
        df = pd.read_csv(h)
        for ax, col in zip(axes, ["train_loss", "val_macro_f1", "val_recall_spoiled"]):
            ax.plot(df["epoch"] + 1, df[col], label=Path(d).name)
    for ax, t in zip(axes, ["training loss", "validation macro-F1", "validation spoiled recall"]):
        ax.set_title(f"{title} {t}".strip()); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    axes[1].legend(); plt.tight_layout(); plt.show()

plot_curves([os.path.join(TEACHER_RUNS, a) for a in ARCHS], "teacher")

## 5. Select the teacher (validation) and evaluate it once on test

In [ ]:
!python -m src.train_teacher select --data {CROPS} --split-file {SPLIT} --out {TEACHER_RUNS} --num-workers {WORKERS}
import json
sel = json.load(open(os.path.join(TEACHER_RUNS, "teacher_selection.json")))
print("Selected teacher:", sel["selected_teacher"])
display(pd.DataFrame(sel["validation_ranking"]).set_index("arch").round(4))
t = sel["test"]
print(f"Test ({sel['n_test_images']} images, {sel['n_test_sessions']} sessions): accuracy {t['accuracy']:.4f} | "
      f"macro-F1 {t['macro_f1']:.4f} (95% CI {t['macro_f1_ci95_sessions'][0]:.3f}-{t['macro_f1_ci95_sessions'][1]:.3f}) | "
      f"spoiled recall {t['recall_positive']:.4f}")

In [ ]:
# Bundle all result files (JSON/CSV, no weights) into one zip to share: runs/results_bundle.zip on Drive
!python -m src.collect_results --runs {RUNS} --out {RUNS}/results_bundle.zip